# LeetCode 2960 — Count Tested Devices After Test Operations bằng Python

Link đề: https://leetcode.com/problems/count-tested-devices-after-test-operations/ · Độ khó: **Easy** · Chủ đề: Array, Simulation, Counting

Skeleton LeetCode:

```python
from typing import List

class Solution:
    def countTestedDevices(self, batteryPercentages: List[int]) -> int:
        ...
```

Ghi chú: đề và các ví dụ được viết theo trí nhớ về đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code, nhưng nên đối chiếu lại với đề gốc nếu muốn khớp từng chữ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: kiểm tra các thiết bị lần lượt từ `0` đến `n - 1`; thiết bị nào còn pin (`> 0`) thì được kiểm tra và **làm hao `1%` pin của mọi thiết bị đứng sau nó** (không xuống dưới `0`).
2. Biết **mô phỏng (simulation)** đúng theo đề bằng hai vòng lặp lồng nhau, và hiểu vì sao nó tốn `O(n²)`.
3. Nhận ra **bước đột phá**: ta không cần thật sự trừ pin; chỉ cần nhớ **"đã kiểm tra bao nhiêu thiết bị trước đó"** (`tested`), vì pin hiện tại của thiết bị `i` là `max(0, b[i] - tested)`.
4. Đi từ cách **mô phỏng trừ pin thật** → **mô phỏng bằng list comprehension** → **một biến đếm `tested`** → **mảng hiệu (difference array) trì hoãn**, và hiểu mối liên hệ giữa chúng.
5. Tránh các lỗi kinh điển: chỉ đếm số thiết bị có pin dương, dùng `>=` thay vì `>`, tăng bộ đếm cả khi không kiểm tra, và sửa luôn mảng đầu vào.

## 2. Đề bài & ví dụ

Cho mảng `batteryPercentages` độ dài `n` (đánh chỉ số từ `0`) là phần trăm pin của `n` thiết bị. Ta kiểm tra **lần lượt** từng thiết bị `i` từ `0` đến `n - 1` như sau:

- Nếu `batteryPercentages[i] > 0`:
  - **tăng** số thiết bị đã kiểm tra thêm `1`;
  - **giảm `1`** phần trăm pin của **mọi thiết bị `j` trong khoảng `[i + 1, n - 1]`**, nhưng không để nhỏ hơn `0`, tức `batteryPercentages[j] = max(0, batteryPercentages[j] - 1)`;
  - chuyển sang thiết bị kế tiếp.
- Ngược lại (pin bằng `0`): chuyển sang thiết bị kế tiếp **mà không kiểm tra** và **không** trừ pin ai.

Trả về **số thiết bị được kiểm tra**.

**Ví dụ 1**

- Input: `batteryPercentages = [1, 1, 2, 1, 3]`
- Output: `3`
- Giải thích: `i = 0`: pin `1 > 0` nên kiểm tra, mảng thành `[1, 0, 1, 0, 2]`. `i = 1`: pin `0`, bỏ qua. `i = 2`: pin `1 > 0` nên kiểm tra, mảng thành `[1, 0, 1, 0, 1]`. `i = 3`: pin `0`, bỏ qua. `i = 4`: pin `1 > 0` nên kiểm tra. Tổng `3`.

**Ví dụ 2**

- Input: `batteryPercentages = [0, 1, 2]`
- Output: `2`
- Giải thích: `i = 0`: pin `0`, bỏ qua. `i = 1`: pin `1 > 0` nên kiểm tra, mảng thành `[0, 1, 1]`. `i = 2`: pin `1 > 0` nên kiểm tra. Tổng `2`.

**Ràng buộc:** `1 <= n == batteryPercentages.length <= 100`; `0 <= batteryPercentages[i] <= 100`.

In [1]:
from typing import List
import random

examples = [
    ([1, 1, 2, 1, 3], 3),
    ([0, 1, 2], 2),
    ([5], 1),
    ([0, 0, 0], 0),
]
for b, expected in examples:
    print("batteryPercentages =", b, "-> kỳ vọng", expected)

batteryPercentages = [1, 1, 2, 1, 3] -> kỳ vọng 3
batteryPercentages = [0, 1, 2] -> kỳ vọng 2
batteryPercentages = [5] -> kỳ vọng 1
batteryPercentages = [0, 0, 0] -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1. Duyệt theo chỉ số: `for i in range(n)`

Khi cần biết **vị trí** `i` (để tác động lên các phần tử phía sau), ta duyệt bằng `for i in range(len(a))`. Ở bài này mỗi `i` quyết định ảnh hưởng đến cả đoạn `i + 1 ... n - 1`.

In [2]:
a = [1, 1, 2, 1, 3]
n = len(a)
for i in range(n):
    print(i, "-> các thiết bị phía sau:", list(range(i + 1, n)))

0 -> các thiết bị phía sau: [1, 2, 3, 4]
1 -> các thiết bị phía sau: [2, 3, 4]
2 -> các thiết bị phía sau: [3, 4]
3 -> các thiết bị phía sau: [4]
4 -> các thiết bị phía sau: []


### 3.2. Cắt lát và `max(0, x)` để chặn dưới

`a[i + 1:]` là phần **sau** chỉ số `i`. `max(0, x)` giữ giá trị không dưới `0`: `max(0, -3) = 0`, `max(0, 4) = 4`. Đề dùng đúng phép này để pin không âm.

In [3]:
a = [1, 1, 2, 1, 3]
i = 0
print(a[i + 1:])                                  # [1, 2, 1, 3]
print([max(0, x - 1) for x in a[i + 1:]])         # [0, 1, 0, 2]
print(max(0, -3), max(0, 4))                      # 0 4

[1, 2, 1, 3]
[0, 1, 0, 2]
0 4


### 3.3. Gán vào lát cắt để sửa một đoạn của list

Có thể gán lại cả đoạn: `a[i + 1:] = [...]`. Số phần tử mới nên bằng số phần tử cũ để độ dài không đổi. Đây là cách ngắn để "trừ `1` cho mọi phần tử phía sau".

In [4]:
a = [1, 1, 2, 1, 3]
a[1:] = [max(0, x - 1) for x in a[1:]]
print(a)                                          # [1, 0, 1, 0, 2]

[1, 0, 1, 0, 2]


### 3.4. Sao chép trước khi sửa

Đề **sửa mảng pin trong khi mô phỏng**. Nếu ta sửa trực tiếp `batteryPercentages` thì mảng của người gọi bị đổi. Nên làm việc trên **bản sao** `b = list(batteryPercentages)` (hoặc `a[:]`) để không có tác dụng phụ.

In [5]:
original = [1, 1, 2]
alias = original            # KHÔNG phải bản sao
copy = list(original)       # bản sao
alias[0] = 99
print(original, copy)       # [99, 1, 2] [1, 1, 2]

[99, 1, 2] [1, 1, 2]


### 3.5. Biến đếm trong vòng lặp

Một biến như `tested = 0` rồi `tested += 1` mỗi khi gặp điều kiện là cách đếm đơn giản nhất. Giá trị của biến **tại thời điểm xét `i`** chính là số thiết bị đã kiểm tra **trước `i`**. Đây là mấu chốt của lời giải tối ưu.

In [6]:
a = [1, 1, 2, 1, 3]
tested = 0
for i, x in enumerate(a):
    print(f"i={i}: trước khi xét, tested={tested}")
    if x > 0:                 # (tạm thời bỏ qua việc pin giảm, chỉ để xem biến đếm tăng thế nào)
        tested += 1

i=0: trước khi xét, tested=0
i=1: trước khi xét, tested=1
i=2: trước khi xét, tested=2
i=3: trước khi xét, tested=3
i=4: trước khi xét, tested=4


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Hãy làm **đúng theo đề** với `[1, 1, 2, 1, 3]` và ghi mảng sau mỗi bước:

| `i` | pin hiện tại `b[i]` | `> 0`? | `tested` sau bước | mảng sau bước |
|---|---|---|---|---|
| 0 | 1 | có | 1 | `[1, 0, 1, 0, 2]` |
| 1 | 0 | không | 1 | `[1, 0, 1, 0, 2]` |
| 2 | 1 | có | 2 | `[1, 0, 1, 0, 1]` |
| 3 | 0 | không | 2 | `[1, 0, 1, 0, 1]` |
| 4 | 1 | có | 3 | `[1, 0, 1, 0, 0]` |

Quan sát: pin hiện tại của thiết bị `i` luôn bằng **pin ban đầu trừ đi số thiết bị đã kiểm tra trước nó** (nhưng không dưới `0`). Ví dụ thiết bị `4` có pin ban đầu `3`, trước nó đã kiểm tra `2` thiết bị (`0` và `2`) nên còn `3 - 2 = 1`. Vậy **không cần trừ thật**, chỉ cần đếm.

In [7]:
b = [1, 1, 2, 1, 3]
tested = 0
for i, x in enumerate(b):
    now = max(0, x - tested)                          # pin hiện tại nếu coi như bị trừ `tested` lần
    print(f"i={i} pin gốc={x} tested trước đó={tested} -> pin hiện tại={now}", "=> kiểm tra" if now > 0 else "=> bỏ qua")
    if now > 0:
        tested += 1
print("tổng:", tested)

i=0 pin gốc=1 tested trước đó=0 -> pin hiện tại=1 => kiểm tra
i=1 pin gốc=1 tested trước đó=1 -> pin hiện tại=0 => bỏ qua
i=2 pin gốc=2 tested trước đó=1 -> pin hiện tại=1 => kiểm tra
i=3 pin gốc=1 tested trước đó=2 -> pin hiện tại=0 => bỏ qua
i=4 pin gốc=3 tested trước đó=2 -> pin hiện tại=1 => kiểm tra
tổng: 3


## 5. Cách 1 — Brute Force / baseline: mô phỏng đúng theo đề

**Ý tưởng.** Sao chép mảng, rồi với mỗi `i` nếu `b[i] > 0` thì `tested += 1` và dùng vòng lặp trong trừ `1` (không dưới `0`) cho mọi `j` từ `i + 1` đến `n - 1`.

In [8]:
def count_tested_simulate(batteryPercentages: List[int]) -> int:
    b = list(batteryPercentages)                      # không sửa mảng của người gọi
    n = len(b)
    tested = 0
    for i in range(n):
        if b[i] > 0:
            tested += 1
            for j in range(i + 1, n):
                b[j] = max(0, b[j] - 1)               # trừ 1, không xuống dưới 0
    return tested


for b, expected in examples:
    print(b, "->", count_tested_simulate(b), "(kỳ vọng", expected, ")")

[1, 1, 2, 1, 3] -> 3 (kỳ vọng 3 )
[0, 1, 2] -> 2 (kỳ vọng 2 )
[5] -> 1 (kỳ vọng 1 )
[0, 0, 0] -> 0 (kỳ vọng 0 )


In [9]:
# Xem mảng thay đổi từng bước (ví dụ 1)
b = [1, 1, 2, 1, 3]
n, tested = len(b), 0
for i in range(n):
    if b[i] > 0:
        tested += 1
        for j in range(i + 1, n):
            b[j] = max(0, b[j] - 1)
    print(f"sau i={i}: mảng={b} tested={tested}")

sau i=0: mảng=[1, 0, 1, 0, 2] tested=1
sau i=1: mảng=[1, 0, 1, 0, 2] tested=1
sau i=2: mảng=[1, 0, 1, 0, 1] tested=2
sau i=3: mảng=[1, 0, 1, 0, 1] tested=2
sau i=4: mảng=[1, 0, 1, 0, 1] tested=3


**Độ phức tạp (Cách 1).** Vòng ngoài `n` lần, vòng trong tối đa `n - 1` lần: **Time `O(n²)`**. Bản sao mảng: **Space `O(n)`**. Với `n <= 100` là `~5 000` phép, nhanh, nhưng ta có thể làm đúng trong `O(n)`.

**Nút thắt.** Mỗi lần kiểm tra ta **trừ pin của cả đoạn phía sau**, trong khi thực chất ta chỉ quan tâm pin của thiết bị `i` **vào lúc đến lượt nó**.

## 6. Ý tưởng thuật toán / bước đột phá

Thiết bị `i` bị trừ `1` **mỗi lần** có một thiết bị `k < i` được kiểm tra. Vậy tại lúc đến lượt `i`, nó đã bị trừ đúng `tested` lần, với `tested` là số thiết bị được kiểm tra trước nó. Pin lúc đó là:

> `max(0, batteryPercentages[i] - tested)`

và nó được kiểm tra khi và chỉ khi `batteryPercentages[i] - tested > 0`, tức `batteryPercentages[i] > tested`. Việc "chặn dưới tại `0`" không thay đổi kết luận `> 0`: nếu `b[i] - tested <= 0` thì pin thực là `0`, không kiểm tra; ngược lại pin thực đúng bằng `b[i] - tested`.

Từ đó, **một biến `tested` và một vòng lặp** là đủ: `O(n)` thời gian, `O(1)` bộ nhớ.

## 7. Các cách tiếp theo

### Cách 2 — Mô phỏng bằng list comprehension / gán lát cắt

**Giải thích.** Vẫn mô phỏng trừ pin thật, nhưng thay vòng lặp trong bằng một dòng: `b[i + 1:] = [max(0, x - 1) for x in b[i + 1:]]`. Cùng độ phức tạp `O(n²)`, chỉ khác cách viết (hữu ích để đối chiếu với Cách 1).

In [10]:
def count_tested_slice(batteryPercentages: List[int]) -> int:
    b = list(batteryPercentages)
    tested = 0
    for i in range(len(b)):
        if b[i] > 0:
            tested += 1
            b[i + 1:] = [max(0, x - 1) for x in b[i + 1:]]    # trừ 1 cho cả đoạn phía sau
    return tested


for b, expected in examples:
    print(b, "->", count_tested_slice(b), "(kỳ vọng", expected, ")")

[1, 1, 2, 1, 3] -> 3 (kỳ vọng 3 )
[0, 1, 2] -> 2 (kỳ vọng 2 )
[5] -> 1 (kỳ vọng 1 )
[0, 0, 0] -> 0 (kỳ vọng 0 )


**Độ phức tạp Cách 2:** **Time `O(n²)`**, **Space `O(n)`**.

### Cách 3 — Một biến đếm `tested` (O(n), O(1))

**Giải thích.** Dựa trên mục 6: duyệt từng `x` trong mảng; nếu `x > tested` thì thiết bị này được kiểm tra (`tested += 1`). Không sửa mảng, không bản sao.

In [11]:
def count_tested_counter(batteryPercentages: List[int]) -> int:
    tested = 0
    for x in batteryPercentages:
        if x > tested:                       # pin gốc - số lần đã bị trừ > 0
            tested += 1
    return tested


for b, expected in examples:
    print(b, "->", count_tested_counter(b), "(kỳ vọng", expected, ")")

[1, 1, 2, 1, 3] -> 3 (kỳ vọng 3 )
[0, 1, 2] -> 2 (kỳ vọng 2 )
[5] -> 1 (kỳ vọng 1 )
[0, 0, 0] -> 0 (kỳ vọng 0 )


**Độ phức tạp Cách 3:** **Time `O(n)`**, **Space `O(1)`**.

### Cách 4 — Mảng hiệu (difference array) "trừ trì hoãn"

**Giải thích.** Đây là kỹ thuật tổng quát để làm **cập nhật đoạn** hiệu quả: thay vì trừ `1` cho cả đoạn `[i + 1, n - 1]`, ta ghi vào `diff[i + 1] += 1` (đánh dấu "từ đây trở đi mỗi phần tử bị trừ thêm `1`"). Khi đến phần tử `j`, ta cộng dồn `pending += diff[j]` để biết **tổng số lần đã bị trừ**. Với đoạn luôn kéo dài đến hết mảng, mảng hiệu thu về đúng một biến đếm (Cách 3), nhưng cách viết này cho thấy **vì sao** Cách 3 đúng, và dùng được khi đoạn cập nhật có điểm kết thúc.

In [12]:
def count_tested_diff(batteryPercentages: List[int]) -> int:
    n = len(batteryPercentages)
    diff = [0] * (n + 1)                     # diff[j]: có thêm bao nhiêu lần "bắt đầu trừ" tại j
    pending = 0                              # tổng số lần đã bị trừ của thiết bị hiện tại
    tested = 0
    for j in range(n):
        pending += diff[j]
        if batteryPercentages[j] - pending > 0:
            tested += 1
            diff[j + 1] += 1                 # từ j + 1 đến hết mảng, mỗi phần tử bị trừ thêm 1
    return tested


for b, expected in examples:
    print(b, "->", count_tested_diff(b), "(kỳ vọng", expected, ")")

# Quan sát diff và pending cho ví dụ 1
b = [1, 1, 2, 1, 3]
diff, pending = [0] * (len(b) + 1), 0
for j in range(len(b)):
    pending += diff[j]
    ok = b[j] - pending > 0
    if ok:
        diff[j + 1] += 1
    print(f"j={j} b[j]={b[j]} pending={pending} pin hiện tại={max(0, b[j] - pending)} kiểm tra={ok} diff={diff}")

[1, 1, 2, 1, 3] -> 3 (kỳ vọng 3 )
[0, 1, 2] -> 2 (kỳ vọng 2 )
[5] -> 1 (kỳ vọng 1 )
[0, 0, 0] -> 0 (kỳ vọng 0 )
j=0 b[j]=1 pending=0 pin hiện tại=1 kiểm tra=True diff=[0, 1, 0, 0, 0, 0]
j=1 b[j]=1 pending=1 pin hiện tại=0 kiểm tra=False diff=[0, 1, 0, 0, 0, 0]
j=2 b[j]=2 pending=1 pin hiện tại=1 kiểm tra=True diff=[0, 1, 0, 1, 0, 0]
j=3 b[j]=1 pending=2 pin hiện tại=0 kiểm tra=False diff=[0, 1, 0, 1, 0, 0]
j=4 b[j]=3 pending=2 pin hiện tại=1 kiểm tra=True diff=[0, 1, 0, 1, 0, 1]


**Độ phức tạp Cách 4:** **Time `O(n)`**, **Space `O(n)`** cho mảng `diff`.

In [13]:
# Code nộp LeetCode cho Cách 3 (ngắn nhất, O(n) thời gian, O(1) bộ nhớ)
from typing import List

class Solution:
    def countTestedDevices(self, batteryPercentages: List[int]) -> int:
        tested = 0
        for x in batteryPercentages:
            if x > tested:
                tested += 1
        return tested

## 8. TRACE từng bước

Ta theo dõi (a) Cách 3 với `[1, 1, 2, 1, 3]` và (b) so sánh với mô phỏng thật để thấy "pin hiện tại = `max(0, b[i] - tested)`".

In [14]:
def trace_counter(b):
    tested = 0
    print("batteryPercentages =", b)
    for i, x in enumerate(b):
        ok = x > tested
        print(f"i={i}: x={x} tested={tested} -> x > tested? {ok!s:5} ", end="")
        if ok:
            tested += 1
            print(f"=> kiểm tra, tested={tested}")
        else:
            print(f"=> bỏ qua, tested={tested}")
    print("=> kết quả:", tested)
    return tested


assert trace_counter([1, 1, 2, 1, 3]) == 3
assert trace_counter([0, 1, 2]) == 2

batteryPercentages = [1, 1, 2, 1, 3]
i=0: x=1 tested=0 -> x > tested? True  => kiểm tra, tested=1
i=1: x=1 tested=1 -> x > tested? False => bỏ qua, tested=1
i=2: x=2 tested=1 -> x > tested? True  => kiểm tra, tested=2
i=3: x=1 tested=2 -> x > tested? False => bỏ qua, tested=2
i=4: x=3 tested=2 -> x > tested? True  => kiểm tra, tested=3
=> kết quả: 3
batteryPercentages = [0, 1, 2]
i=0: x=0 tested=0 -> x > tested? False => bỏ qua, tested=0
i=1: x=1 tested=0 -> x > tested? True  => kiểm tra, tested=1
i=2: x=2 tested=1 -> x > tested? True  => kiểm tra, tested=2
=> kết quả: 2


In [15]:
def trace_compare(b):
    """Chạy mô phỏng thật và công thức max(0, b[i] - tested), so sánh từng bước."""
    cur = list(b)
    tested = 0
    for i in range(len(b)):
        formula = max(0, b[i] - tested)              # công thức (không sửa mảng)
        real = cur[i]                                 # pin thật sau các lần trừ
        assert real == formula, (i, real, formula)
        print(f"i={i}: pin thật={real} | công thức max(0, {b[i]} - {tested})={formula} | khớp")
        if real > 0:
            tested += 1
            for j in range(i + 1, len(cur)):
                cur[j] = max(0, cur[j] - 1)
    return tested


assert trace_compare([1, 1, 2, 1, 3]) == 3

i=0: pin thật=1 | công thức max(0, 1 - 0)=1 | khớp
i=1: pin thật=0 | công thức max(0, 1 - 1)=0 | khớp
i=2: pin thật=1 | công thức max(0, 2 - 1)=1 | khớp
i=3: pin thật=0 | công thức max(0, 1 - 2)=0 | khớp
i=4: pin thật=1 | công thức max(0, 3 - 2)=1 | khớp


Bảng tóm tắt (Cách 3, `[1, 1, 2, 1, 3]`):

| `i` | `x` | `tested` trước | `x > tested`? | `tested` sau |
|---|---|---|---|---|
| 0 | 1 | 0 | có | 1 |
| 1 | 1 | 1 | không | 1 |
| 2 | 2 | 1 | có | 2 |
| 3 | 1 | 2 | không | 2 |
| 4 | 3 | 2 | có | 3 |

## 9. Code nộp LeetCode

Bản nộp dùng Cách 3 (một biến đếm). Đúng class `Solution`, đúng method `countTestedDevices(self, batteryPercentages: List[int]) -> int`.

In [16]:
from typing import List

class Solution:
    def countTestedDevices(self, batteryPercentages: List[int]) -> int:
        tested = 0                                   # số thiết bị đã kiểm tra trước thiết bị hiện tại
        for x in batteryPercentages:
            if x > tested:                           # pin còn lại = x - tested > 0
                tested += 1
        return tested


sol = Solution()
for b, expected in examples:
    print(b, "->", sol.countTestedDevices(b))

[1, 1, 2, 1, 3] -> 3
[0, 1, 2] -> 2
[5] -> 1
[0, 0, 0] -> 0


## 10. Vì sao thuật toán đúng

- **Pin hiện tại có công thức đóng.** Thiết bị `i` bị trừ `1` mỗi lần có thiết bị `k < i` được kiểm tra, nên lúc đến lượt `i` nó bị trừ đúng `tested` lần, với `tested` là số thiết bị được kiểm tra trong `0..i-1`. Pin hiện tại là `max(0, b[i] - tested)`.
- **Điều kiện kiểm tra.** `i` được kiểm tra ⇔ pin hiện tại `> 0` ⇔ `b[i] - tested > 0` (nếu `b[i] - tested <= 0` thì pin `max(0, ·) = 0`, không kiểm tra). Đây chính là `x > tested`.
- **Quy nạp.** Giả sử sau khi xử lý `i` thiết bị đầu, biến `tested` đúng bằng số thiết bị đã kiểm tra thật. Với thiết bị `i`, điều kiện `x > tested` quyết định đúng việc nó được kiểm tra, và `tested` tăng đúng khi đó, nên giả thiết còn đúng sau bước `i`.
- **Chặn dưới tại `0` vô hại.** Thiết bị đã `0` pin tiếp tục `0`; trừ thêm không làm điều kiện `> 0` đổi từ sai sang đúng.
- **Cận trên.** Thiết bị kiểm tra thứ `c` có `b >= c` nên đáp án không vượt `max(b)` và không vượt `n`.

## 11. Phân tích Time / Space Complexity

Gọi `n` là số thiết bị.

- **Cách 1 (mô phỏng hai vòng):** Time `O(n²)`, Space `O(n)`.
- **Cách 2 (gán lát cắt):** Time `O(n²)`, Space `O(n)`.
- **Cách 3 (biến đếm):** Time `O(n)`, Space `O(1)`.
- **Cách 4 (mảng hiệu):** Time `O(n)`, Space `O(n)`.
- Phải đọc mọi thiết bị nên `O(n)` là tối ưu. Ở đây `n <= 100`, mọi cách đều chạy tức thì; ý nghĩa của bài là học cách **thay thao tác trên cả đoạn bằng một bộ đếm**.

## 12. Bộ test

In [17]:
sol = Solution()
impls = {
    "simulate": count_tested_simulate,
    "slice": count_tested_slice,
    "counter": count_tested_counter,
    "diff": count_tested_diff,
    "submit": sol.countTestedDevices,
}

for b, expected in examples:
    for key, fn in impls.items():
        assert fn(list(b)) == expected, (key, b, fn(list(b)), expected)
print("PASS ví dụ của đề (mọi cách)")

PASS ví dụ của đề (mọi cách)


### 12.1. Các trường hợp biên

In [18]:
def check_all(b, expected, label):
    for key, fn in impls.items():
        got = fn(list(b))
        assert got == expected, (label, key, got, expected)
    print("PASS", label, "->", expected)


check_all([0], 0, "một thiết bị hết pin")
check_all([1], 1, "một thiết bị còn pin")
check_all([100], 1, "một thiết bị pin 100")
check_all([1, 1], 1, "thiết bị thứ hai bị trừ về 0")
check_all([1, 2], 2, "thiết bị thứ hai pin 2 còn 1 > 0")
check_all([1, 2, 3, 4, 5], 5, "tăng dần: tất cả được kiểm tra")
check_all([5, 4, 3, 2, 1], 3, "giảm dần: 5, 4->3, 3->1... chỉ kiểm tra 3 thiết bị đầu")
check_all([0, 0, 0, 5], 1, "toàn 0 rồi 5: chỉ thiết bị cuối, không ai trừ pin")
check_all([1] * 100, 1, "100 thiết bị pin 1: chỉ cái đầu")
check_all(list(range(1, 101)), 100, "n = 100 tăng dần 1..100: tất cả")
check_all([100] * 100, 100, "n = 100, pin 100: thiết bị thứ i còn 100 - i >= 1")
check_all([0] * 100, 0, "n = 100 toàn 0")

PASS một thiết bị hết pin -> 0
PASS một thiết bị còn pin -> 1
PASS một thiết bị pin 100 -> 1
PASS thiết bị thứ hai bị trừ về 0 -> 1
PASS thiết bị thứ hai pin 2 còn 1 > 0 -> 2
PASS tăng dần: tất cả được kiểm tra -> 5
PASS giảm dần: 5, 4->3, 3->1... chỉ kiểm tra 3 thiết bị đầu -> 3
PASS toàn 0 rồi 5: chỉ thiết bị cuối, không ai trừ pin -> 1
PASS 100 thiết bị pin 1: chỉ cái đầu -> 1
PASS n = 100 tăng dần 1..100: tất cả -> 100
PASS n = 100, pin 100: thiết bị thứ i còn 100 - i >= 1 -> 100
PASS n = 100 toàn 0 -> 0


### 12.2. Tính chất và phản ví dụ

In [19]:
# Tính chất 1: đáp án <= số thiết bị có pin dương, <= n, <= max(pin)
random.seed(11)
for _ in range(500):
    b = [random.randint(0, 100) for _ in range(random.randint(1, 100))]
    r = sol.countTestedDevices(b)
    assert r <= sum(1 for x in b if x > 0) and r <= len(b) and r <= max(b)
print("PASS đáp án <= số thiết bị dương, <= n, <= max(pin)")

# Tính chất 2: thêm số 0 vào đầu mảng không đổi đáp án (không kiểm tra, không trừ ai)
for _ in range(300):
    b = [random.randint(0, 20) for _ in range(random.randint(1, 30))]
    assert sol.countTestedDevices([0] + b) == sol.countTestedDevices(b)
print("PASS thêm số 0 ở đầu không đổi đáp án")

# Tính chất 3: công thức pin hiện tại = max(0, b[i] - tested) khớp mô phỏng ở mọi bước (trace_compare)
for _ in range(100):
    b = [random.randint(0, 10) for _ in range(random.randint(1, 20))]
    cur, tested = list(b), 0
    for i in range(len(b)):
        assert cur[i] == max(0, b[i] - tested)
        if cur[i] > 0:
            tested += 1
            cur[i + 1:] = [max(0, x - 1) for x in cur[i + 1:]]
print("PASS công thức pin hiện tại khớp mô phỏng ở mọi bước")

# Tính chất 4: đầu vào không bị sửa
b = [1, 1, 2, 1, 3]
copy = list(b)
for key, fn in impls.items():
    fn(b)
assert b == copy
print("PASS đầu vào không bị sửa")

# Phản ví dụ 1: chỉ đếm thiết bị có pin dương
def sai_dem_duong(b):
    return sum(1 for x in b if x > 0)

assert sai_dem_duong([1, 1, 2, 1, 3]) == 5 != 3
print("PASS phản ví dụ: đếm pin dương cho 5 thay vì 3")

# Phản ví dụ 2: dùng >= thay vì >
def sai_ge(b):
    tested = 0
    for x in b:
        if x >= tested:
            tested += 1
    return tested

assert sai_ge([1, 1]) == 2 and sol.countTestedDevices([1, 1]) == 1
print("PASS phản ví dụ: x >= tested cho", sai_ge([1, 1]), "thay vì 1")

PASS đáp án <= số thiết bị dương, <= n, <= max(pin)
PASS thêm số 0 ở đầu không đổi đáp án
PASS công thức pin hiện tại khớp mô phỏng ở mọi bước
PASS đầu vào không bị sửa
PASS phản ví dụ: đếm pin dương cho 5 thay vì 3
PASS phản ví dụ: x >= tested cho 2 thay vì 1


### 12.3. Đối chiếu ngẫu nhiên (có seed)

In [20]:
random.seed(2960)
for trial in range(3000):
    n = random.randint(1, 100)
    hi = random.choice([1, 3, 10, 100])
    b = [random.randint(0, hi) for _ in range(n)]
    ref = count_tested_simulate(list(b))
    for key, fn in impls.items():
        assert fn(list(b)) == ref, (key, b)
print("PASS 3000 bộ ngẫu nhiên - mọi cách khớp nhau")

PASS 3000 bộ ngẫu nhiên - mọi cách khớp nhau


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Chỉ đếm thiết bị có pin dương

In [21]:
def sai_duong(b):
    return sum(1 for x in b if x > 0)


print("sai  :", sai_duong([1, 1, 2, 1, 3]), "| đúng:", Solution().countTestedDevices([1, 1, 2, 1, 3]))

sai  : 5 | đúng: 3


**Vì sao sai:** pin của một thiết bị **giảm** mỗi khi có thiết bị đứng trước được kiểm tra. Thiết bị pin `1` ở vị trí `1` bị trừ về `0` sau khi thiết bị `0` được kiểm tra. Không thể chỉ nhìn pin ban đầu.

### Lỗi 2: Dùng `>=` thay vì `>`

In [22]:
def sai_ge_2(b):
    tested = 0
    for x in b:
        if x >= tested:
            tested += 1
    return tested


print("sai  :", sai_ge_2([1, 1]), "| đúng:", Solution().countTestedDevices([1, 1]))

sai  : 2 | đúng: 1


**Vì sao sai:** pin hiện tại là `x - tested`, và đề yêu cầu `> 0`, tức `x > tested`. Với `x = tested` pin hiện tại bằng `0`, **không** được kiểm tra. `>=` sai khi `x` đúng bằng `tested` (lệch một đơn vị).

### Lỗi 3: Tăng bộ đếm cho mọi thiết bị

In [23]:
def sai_tang_moi_lan(b):
    tested = 0
    for x in b:
        if x > tested:
            pass
        tested += 1                                # tăng cả khi KHÔNG kiểm tra
    return tested


print("sai  :", sai_tang_moi_lan([0, 1, 2]), "| đúng:", Solution().countTestedDevices([0, 1, 2]))

sai  : 3 | đúng: 2


**Vì sao sai:** chỉ thiết bị **được kiểm tra** mới làm hao pin những thiết bị phía sau. Thiết bị pin `0` bị bỏ qua và **không** trừ pin ai, nên `tested` chỉ được tăng bên trong `if`.

### Lỗi 4: Sửa luôn mảng đầu vào

In [24]:
def sai_sua_dau_vao(b):
    tested = 0
    for i in range(len(b)):
        if b[i] > 0:
            tested += 1
            for j in range(i + 1, len(b)):
                b[j] = max(0, b[j] - 1)                # sửa trực tiếp mảng của người gọi
    return tested


data = [1, 1, 2, 1, 3]
print("kết quả:", sai_sua_dau_vao(data), "| mảng sau khi gọi:", data)

kết quả: 3 | mảng sau khi gọi: [1, 0, 1, 0, 1]


**Vì sao không tốt:** hàm trả về đúng số, nhưng **làm hỏng dữ liệu của người gọi** (mảng ban đầu `[1, 1, 2, 1, 3]` bị đổi). Trong bài mà ta còn cần dùng lại mảng (ví dụ gọi hai lần để so sánh) sẽ cho kết quả sai. Hãy sao chép (`list(b)`) hoặc dùng Cách 3, vốn không sửa mảng.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy "mỗi lần làm A thì **trừ `1` cho mọi phần tử phía sau**" → đừng trừ thật; dùng **một biến đếm số lần đã trừ**.
- Công thức nhớ: **pin hiện tại = `max(0, b[i] - tested)`**, kiểm tra khi `b[i] > tested`.
- Tổng quát: cập nhật đoạn nhiều lần + truy vấn từng điểm theo thứ tự → **mảng hiệu (difference array)** + cộng dồn.
- Nếu đề cho phép `n` lớn (`10⁵`), bản mô phỏng `O(n²)` sẽ quá chậm; luôn nghĩ tới bước đột phá "đếm thay vì trừ".
- `if x > 0` là kiểm tra **pin gốc**; đề lại hỏi pin **sau khi bị trừ**. Đọc kỹ thời điểm kiểm tra.
- Khi mô phỏng nên làm việc trên **bản sao** nếu đề không cho phép sửa đầu vào.

## 15. Biến thể của bài

1. Mỗi lần kiểm tra trừ **`d`** phần trăm thay vì `1` (giữ `max(0, ...)`).
2. Chỉ trừ pin của **`r` thiết bị kế tiếp** (đoạn `[i + 1, i + r]`) thay vì cả phần còn lại.
3. Trả về **danh sách chỉ số** các thiết bị được kiểm tra.
4. Duyệt từ **phải sang trái** thay vì trái sang phải.
5. Có thêm **sạc**: sau mỗi thiết bị được kiểm tra, thiết bị đứng ngay sau được cộng thêm `1`.

Thử biến thể 1, 2 và 3:

In [25]:
def tested_with_drop(b, d):
    tested = 0
    for x in b:
        if x - tested * d > 0:               # mỗi lần kiểm tra trước đó trừ d
            tested += 1
    return tested


def tested_with_range(b, r):
    """Mỗi lần kiểm tra chỉ trừ 1 cho r thiết bị kế tiếp: dùng mảng hiệu có điểm kết thúc."""
    n = len(b)
    diff = [0] * (n + r + 2)
    pending = tested = 0
    for j in range(n):
        pending += diff[j]
        if b[j] - pending > 0:
            tested += 1
            diff[j + 1] += 1                 # bắt đầu trừ từ j + 1
            diff[j + r + 1] -= 1             # hết trừ sau j + r
    return tested


def tested_indices(b):
    tested, ans = 0, []
    for i, x in enumerate(b):
        if x > tested:
            tested += 1
            ans.append(i)
    return ans


def tested_with_range_sim(b, r):
    cur, tested = list(b), 0
    for i in range(len(cur)):
        if cur[i] > 0:
            tested += 1
            for j in range(i + 1, min(len(cur), i + r + 1)):
                cur[j] = max(0, cur[j] - 1)
    return tested


print("1. d = 2:", tested_with_drop([5, 4, 3, 2, 1], 2))                 # 2: 5>0 kiểm tra; 4-2=2>0 kiểm tra; 3-4<0 dừng
print("3. chỉ số được kiểm tra:", tested_indices([1, 1, 2, 1, 3]))        # [0, 2, 4]
random.seed(1)
for _ in range(500):
    b = [random.randint(0, 6) for _ in range(random.randint(1, 15))]
    r = random.randint(1, 5)
    assert tested_with_range(b, r) == tested_with_range_sim(b, r)
print("2. mảng hiệu có điểm kết thúc khớp mô phỏng")
assert tested_with_drop([5, 4, 3, 2, 1], 1) == Solution().countTestedDevices([5, 4, 3, 2, 1])
assert tested_indices([1, 1, 2, 1, 3]) == [0, 2, 4]

1. d = 2: 2
3. chỉ số được kiểm tra: [0, 2, 4]
2. mảng hiệu có điểm kết thúc khớp mô phỏng


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng hai vòng | trừ thật cho đoạn phía sau | `O(n²)` | `O(n)` | hiểu đề, kiểm chứng |
| 2. Gán lát cắt / comprehension | cùng ý, viết gọn | `O(n²)` | `O(n)` | tiện đối chiếu |
| 3. Một biến đếm `tested` | pin hiện tại = `x - tested` | `O(n)` | `O(1)` | **cách nộp bài** |
| 4. Mảng hiệu (difference array) | "trừ trì hoãn" + cộng dồn | `O(n)` | `O(n)` | khi cập nhật đoạn có điểm kết thúc |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `max(0, 2 - 5)`, `[3, 1, 2][1:]`, `Solution().countTestedDevices([2, 2, 2])`, `Solution().countTestedDevices([0, 5, 0, 5])`, `Solution().countTestedDevices([3, 3, 3, 3])`.

**Bài 2 (tính tay).** Với `batteryPercentages = [2, 1, 3, 0, 2]`: ghi bảng từng bước như mục 4 (mảng sau mỗi bước, `tested`), rồi so với công thức `x > tested`.

**Bài 3 (điền TODO).** Hoàn thành `count_tested_todo(b)` theo Cách 3.

In [26]:
def count_tested_todo(b):
    tested = 0
    for x in b:
        # TODO 1: nếu pin gốc x lớn hơn số thiết bị đã kiểm tra thì
        #     TODO 2: tăng tested lên 1
        pass
    return tested

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `count_tested_simulate2(b)` mô phỏng đúng theo đề (trên bản sao mảng, hai vòng lặp, `max(0, ...)`) mà không nhìn code mẫu. Kiểm tra với bốn ví dụ trong `examples`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `last_tested_index(b)` trả về chỉ số của thiết bị **được kiểm tra cuối cùng** (hoặc `-1` nếu không có). (b) Viết `min_initial_battery_for_all(n)` trả về mảng pin **nhỏ nhất theo tổng** để **cả `n` thiết bị đều được kiểm tra** (đoán rồi kiểm chứng bằng `countTestedDevices`). (c) Viết `max_tested_after_removing_one(b)`: bỏ đúng **một** thiết bị khỏi mảng sao cho số thiết bị được kiểm tra là **lớn nhất**; trả về giá trị lớn nhất đó.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `0`, `[1, 2]`, `[2, 2, 2]`: thiết bị 0 kiểm tra (tested = 1), thiết bị 1: `2 > 1` kiểm tra (tested = 2), thiết bị 2: `2 > 2` sai nên kết quả `2`; `[0, 5, 0, 5]`: `0` bỏ, `5 > 0` kiểm tra, `0` bỏ, `5 > 1` kiểm tra nên `2`; `[3, 3, 3, 3]`: `3 > 0`, `3 > 1`, `3 > 2`, `3 > 3` sai nên `3`. Bài 3: `if x > tested: tested += 1`.
- **Mức 2 (vừa):** Bài 2: `[2,1,3,0,2]`: `i=0`: `2 > 0` kiểm tra, `tested = 1`; `i=1`: `1 > 1`? không; `i=2`: `3 > 1` kiểm tra, `tested = 2`; `i=3`: `0 > 2`? không; `i=4`: `2 > 2`? không. Kết quả `2`. Bài 4: copy Cách 1, nhớ `list(b)`.
- **Mức 3 (sâu):** Bài 5(a): dùng `ans = -1`, cập nhật `ans = i` mỗi lần kiểm tra. 5(b): thiết bị `i` cần pin `>= i + 1`, nên mảng nhỏ nhất theo từng phần tử là `[1, 2, ..., n]`. 5(c): thử bỏ từng chỉ số `k` (`b[:k] + b[k+1:]`), tính đáp án, lấy `max` (`O(n²)` là đủ).

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [27]:
# Bài 1
assert max(0, 2 - 5) == 0 and [3, 1, 2][1:] == [1, 2]
assert Solution().countTestedDevices([2, 2, 2]) == 2
assert Solution().countTestedDevices([0, 5, 0, 5]) == 2
assert Solution().countTestedDevices([3, 3, 3, 3]) == 3

# Bài 2
assert Solution().countTestedDevices([2, 1, 3, 0, 2]) == 2
assert count_tested_simulate([2, 1, 3, 0, 2]) == 2

# Bài 3
def count_tested_todo_ans(b):
    tested = 0
    for x in b:
        if x > tested:
            tested += 1
    return tested

for b, expected in examples:
    assert count_tested_todo_ans(b) == expected

# Bài 4
def count_tested_simulate2(b):
    cur = list(b)
    tested = 0
    for i in range(len(cur)):
        if cur[i] > 0:
            tested += 1
            for j in range(i + 1, len(cur)):
                cur[j] = max(0, cur[j] - 1)
    return tested

for b, expected in examples:
    assert count_tested_simulate2(b) == expected

# Bài 5(a)
def last_tested_index(b):
    tested, ans = 0, -1
    for i, x in enumerate(b):
        if x > tested:
            tested += 1
            ans = i
    return ans

assert last_tested_index([1, 1, 2, 1, 3]) == 4
assert last_tested_index([0, 0]) == -1
assert last_tested_index([1, 1, 1]) == 0

# Bài 5(b)
def min_initial_battery_for_all(n):
    return list(range(1, n + 1))

for n in range(1, 30):
    arr = min_initial_battery_for_all(n)
    assert Solution().countTestedDevices(arr) == n
    for i in range(n):                           # giảm bất kỳ phần tử nào xuống 1 đơn vị thì không còn đủ
        if arr[i] > 0:
            arr2 = list(arr)
            arr2[i] -= 1
            assert Solution().countTestedDevices(arr2) < n

# Bài 5(c)
def max_tested_after_removing_one(b):
    return max(Solution().countTestedDevices(b[:k] + b[k + 1:]) for k in range(len(b)))

assert max_tested_after_removing_one([1, 1, 2, 1, 3]) == 3
assert max_tested_after_removing_one([1, 1, 1]) == 1
assert max_tested_after_removing_one([1, 1, 2]) == 2           # bỏ một số 1 -> [1, 2]: cả hai
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (đếm thay vì trừ thật)

```python
from typing import List

class Solution:
    def countTestedDevices(self, batteryPercentages: List[int]) -> int:
        tested = 0
        for x in batteryPercentages:
            if x > tested:
                tested += 1
        return tested
```

**Checklist**

- [ ] Hiểu pin hiện tại của thiết bị `i` là `max(0, b[i] - tested)`, không phải pin gốc?
- [ ] Điều kiện kiểm tra là `x > tested` (nghiêm ngặt), không phải `>=`?
- [ ] `tested` chỉ tăng **bên trong** `if` (thiết bị pin `0` không trừ pin ai)?
- [ ] Nếu mô phỏng: đã sao chép mảng (`list(b)`) và vòng trong chạy `j` từ `i + 1` đến `n - 1`?
- [ ] Đã thử: toàn `0`, `n = 1`, tăng dần `1..n` (tất cả kiểm tra), toàn `1` (chỉ thiết bị đầu), giá trị bằng đúng `tested`?
- [ ] Skeleton đúng `class Solution` / `def countTestedDevices(self, batteryPercentages: List[int]) -> int`?